# Methodology Notes: WikiRate/Wikidata Company Enrichment

This notebook explains the methodology behind the optional company-level enrichment layer. It is written for review and interpretation, not as the main technical demo.

## Purpose

The core Ethical Product Analyzer score comes from product-level evidence: Open Food Facts, Open Beauty Facts, Open Products Facts, and the label-based scoring engine.

WikiRate is used only as an optional company-level enrichment layer for Social, Governance, and Ethics. It should remain separate from `label_mapping.csv`, because `label_mapping.csv` describes product-level labels while WikiRate describes company-level evidence.

The enrichment returns a preview and should be inspected before being integrated into final scores.

## Why Wikidata Is Used as a Bridge

Open*Facts usually provides brand names, not parent companies. WikiRate usually stores ESG information at the company level.

Wikidata helps connect these two layers:

`Open*Facts brand -> Wikidata brand/company entity -> P127/P749 parent or owner company -> WikiRate company lookup`

Low-confidence matches, ambiguous entities, and unrelated entities do not receive score adjustments.

In [9]:
from pathlib import Path
import importlib
import sys

PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/Users/khadija/Desktop/WBS Coding School/ESG Project"),
]
PROJECT_ROOT = next(path for path in PROJECT_ROOT_CANDIDATES if (path / "wikirate_lookup.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import wikirate_lookup
importlib.reload(wikirate_lookup)
wikirate_lookup.clear_caches()

from wikirate_lookup import enrich_brand_with_company_esg, apply_company_adjustments

print(f"Loaded wikirate_lookup from: {wikirate_lookup.__file__}")

Loaded wikirate_lookup from: /Users/khadija/Desktop/WBS Coding School/ESG Project/wikirate_lookup.py


## WikiRate Metric Lookup

After a company is resolved, WikiRate is queried for selected company-level ESG metrics. Company pages alone are not enough; metric answers are queried through company answer cards:

```text
/{Company_Card}+Answers.json?filter[metric_keyword]=...
```

The API client uses caching, retry logic, and short backoff sleeps. HTTP 429 rate limits are marked as failed requests and do not change scores.

In [10]:
result = enrich_brand_with_company_esg("Nutella", debug=True)
result["wikirate"].get("debug", [])[:3]

WikiRate company card URL: https://wikirate.org/Ferrero_SpA.json?api_key=REDACTED

Searched metric: human_rights_policy
  keywords: Human Rights Policy, Human Rights
  URL: https://wikirate.org/Ferrero_SpA+Answers.json?api_key=REDACTED&limit=20&filter%5Bmetric_keyword%5D=Human+Rights+Policy
  HTTP status: 200
  Answers returned: 0
  URL: https://wikirate.org/Ferrero_SpA+Answers.json?api_key=REDACTED&limit=20&filter%5Bmetric_keyword%5D=Human+Rights
  HTTP status: 200
  Answers returned: 0

Searched metric: modern_slavery_statement
  keywords: Modern Slavery Statement
  URL: https://wikirate.org/Ferrero_SpA+Answers.json?api_key=REDACTED&limit=20&filter%5Bmetric_keyword%5D=Modern+Slavery+Statement
  HTTP status: 200
  Answers returned: 6
  Selected metric card: Business & Human Rights Resource Centre+Modern Slavery Statement
  Selected year/value/classification: 2021 / Yes - UK Modern Slavery Act / positive

Searched metric: anti_corruption_policy
  keywords: Anti-Corruption Policy, Anti-

[{'metric_key': 'human_rights_policy',
  'keyword': 'Human Rights Policy',
  'api_url': 'https://wikirate.org/Ferrero_SpA+Answers.json?api_key=REDACTED&limit=20&filter%5Bmetric_keyword%5D=Human+Rights+Policy',
  'http_status': 200,
  'answers_returned': 0,
  'failed': False,
  'reason': None,
  'cached': False},
 {'metric_key': 'human_rights_policy',
  'keyword': 'Human Rights',
  'api_url': 'https://wikirate.org/Ferrero_SpA+Answers.json?api_key=REDACTED&limit=20&filter%5Bmetric_keyword%5D=Human+Rights',
  'http_status': 200,
  'answers_returned': 0,
  'failed': False,
  'reason': None,
  'cached': False},
 {'metric_key': 'modern_slavery_statement',
  'keyword': 'Modern Slavery Statement',
  'api_url': 'https://wikirate.org/Ferrero_SpA+Answers.json?api_key=REDACTED&limit=20&filter%5Bmetric_keyword%5D=Modern+Slavery+Statement',
  'http_status': 200,
  'answers_returned': 6,
  'failed': False,
  'reason': None,
  'cached': False}]

## Scoring Philosophy

- Missing information gives no reward and no penalty.
- Failed requests, including HTTP 429, are not scored.
- Positive policy evidence gives only small bonuses and means disclosure or policy evidence, not proof of perfect ethical behavior.
- Negative controversy-style metrics can create penalties only when explicitly present.
- A controversy metric is treated as a controversy signal, not a legal conclusion.
- Severe controversy evidence suppresses positive policy bonuses in the same pillar.
- The enrichment should be inspected before future integration into the main scoring engine.

## Caps and Integration Boundary

- Maximum positive company-level adjustment per pillar: `+10`
- Maximum negative company-level adjustment per pillar: `-15`
- Final preview scores are clamped between `0` and `100`

For now, company adjustments are returned separately as `company_adjustments` and `updated_scores_preview`. They are not automatically merged into the main product score.

In [11]:
base_scores = {
    "environmental": 70,
    "social": 55,
    "governance": 50,
    "ethics": 60,
}

enrichment = enrich_brand_with_company_esg("Nutella", base_scores=base_scores)

{
    "base_scores": enrichment["base_scores"],
    "company_adjustments": enrichment["company_adjustments"],
    "updated_scores_preview": enrichment["updated_scores_preview"],
    "explanations": enrichment["explanations"],
    "warnings": enrichment["warnings"],
}


{'base_scores': {'environmental': 70,
  'social': 55,
  'governance': 50,
  'ethics': 60},
 'company_adjustments': {'social': 0, 'governance': 9, 'ethics': 3},
 'updated_scores_preview': {'environmental': 70,
  'social': 55,
  'governance': 59,
  'ethics': 63},
 'explanations': ['Business & Human Rights Resource Centre+Modern Slavery Statement found on WikiRate: Ethics +3',
  'Business & Human Rights Resource Centre+Modern Slavery Statement found on WikiRate: Governance +2',
  'Walk Free+MSA supply chain disclosure found on WikiRate: Governance +3',
  'Walk Free+MSA whistleblowing mechanism (binary) found on WikiRate: Governance +4'],
 'warnings': []}